# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Streaming_Session_Analytics

**Input file:** `09_Streaming_Sessions.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("09_Streaming_Sessions.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["watch_minutes"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of watch_minutes.

In [ ]:
t=    session_id user_id content_type  ... subscription completion_pct session_date
0      SS00001   U0173       Sports  ...      Premium           71.1   2025-05-10
1      SS00002   U0124       Series  ...        Basic           17.0   2025-01-10
2      SS00003   U0176       Series  ...         Free           70.6   2025-06-17
3      SS00004   U0023  Documentary  ...         Free           56.8   2025-08-12
4      SS00005   U0124         News  ...      Premium           35.2   2026-01-10
..         ...     ...          ...  ...          ...            ...          ...
397    SS00398   U0014       Sports  ...        Basic           34.2   2025-05-20
398    SS00399   U0033       Series  ...      Premium           88.7   2026-05-26
399    SS00400   U0203       Sports  ...      Premium           67.2   2026-04-27
400    SS00021   U0217        Movie  ...        Basic           52.7   2025-11-11
401    SS00066   U0011         News  ...         Free           63.7   2025-04-07

[402 rows x 10 columns]
threshold=df["watch_minutes"].quantile(.75)
print(df[df["watch_minutes"]>threshold].sort_values("watch_minutes",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total watch_minutes, and average completion_pct for each session_id.

In [ ]:
print(df.groupby("session_id",dropna=False).agg(count=("watch_minutes","count"),total=("watch_minutes","sum"),avg=("completion_pct","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze watch_minutes by session_id and user_id; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["session_id","user_id"],dropna=False).agg(count=("watch_minutes","count"),total=("watch_minutes","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["watch_minutes"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="session_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("session_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["session_id"]=x["session_id"].fillna("Unknown"); print(x["session_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["watch_minutes"]=x["watch_minutes"].fillna(x["watch_minutes"].median()); print(x["watch_minutes"].isna().sum())

## Q11. String search

Find records where session_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["session_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by watch_minutes descending and completion_pct ascending.

In [ ]:
print(df.sort_values(["watch_minutes","completion_pct"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on watch_minutes using its median as the threshold.

In [ ]:
x=df.copy(); med=x["watch_minutes"].median(); x["segment"]=np.where(x["watch_minutes"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("session_id")["watch_minutes"].transform("mean"); print(x[["session_id","watch_minutes","group_avg"]].head(25))

## Q15. Group share

Calculate each session_id's share of total watch_minutes.

In [ ]:
g=df.groupby("session_id",dropna=False)["watch_minutes"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average watch_minutes by session_id and user_id.

In [ ]:
print(pd.pivot_table(df,index="session_id",columns="user_id",values="watch_minutes",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("session_id")[["watch_minutes","completion_pct"]].mean().reset_index(); print(w.melt(id_vars="session_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where watch_minutes is negative or missing.

In [ ]:
print("Invalid:",df["watch_minutes"].isna().sum()+(df["watch_minutes"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in watch_minutes.

In [ ]:
q1=df["watch_minutes"].quantile(.25); q3=df["watch_minutes"].quantile(.75); iqr=q3-q1; print(df[df["watch_minutes"]>q3+1.5*iqr][["watch_minutes"]].head(30))

## Q20. Top group

Find the category in session_id with the highest total watch_minutes.

In [ ]:
print(df.groupby("session_id")["watch_minutes"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in session_id with the lowest average watch_minutes.

In [ ]:
print(df.groupby("session_id")["watch_minutes"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of watch_minutes for each session_id.

In [ ]:
print(df.groupby("session_id")["watch_minutes"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify watch_minutes relative to its median.

In [ ]:
x=df.copy(); med=x["watch_minutes"].median(); x["classification"]=x["watch_minutes"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["session_id"].dropna().unique())}}; x=df.copy(); x["session_id_code"]=x["session_id"].map(codes); print(x[["session_id","session_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null watch_minutes, non-null session_id, and values above the median watch_minutes.

In [ ]:
m=df["watch_minutes"].median(); x=df[df["watch_minutes"].notna()&df["session_id"].notna()&(df["watch_minutes"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("09_Streaming_Sessions.csv",chunksize=50):
    total += chunk["watch_minutes"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final session_id-level report with count, total watch_minutes, average completion_pct, and rank by total watch_minutes.

In [ ]:
r=df.groupby("session_id",dropna=False).agg(records=("watch_minutes","count"),total=("watch_minutes","sum"),average=("completion_pct","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.